# 第八章 · 在T4上做DPO

[![在Colab中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zxh0/vibe-writing/blob/main/toyllm_book2/code/book/ch08/colab.ipynb)

和第六章的笔记本一样，只做杂事：挂载Google Drive、拉代码、准备数据，然后调用`book/ch08/`里的脚本。

**起点是第七章训好的SFT模型**，要在Drive的`我的云端硬盘/toyllm2/runs/sft/model/`下（第七章的笔记本训完就存在那里）。策略模型和参考模型都从它开始。

**数据不用从Mac上传**：第一次运行时，在Colab上下载UltraFeedback（约230MB），用SFT模型目录里的分词器分词，
结果存一份到Drive的`toyllm2/data/dpo/`。以后再运行，直接从Drive复制。

**用法**：菜单“代码执行程序 → 更改运行时类型”选T4 GPU，改好第一个格子里的配置，然后“全部运行”。
断线以后重新连上，再“全部运行”一次，`dpo.py`带着`--resume`，会从上一个checkpoint接着训。

In [ ]:
# ---- 配置 ----------------------------------------------------------------
REPO_URL = "https://github.com/zxh0/vibe-writing.git"
BRANCH = "main"
SUBDIR = "toyllm_book2/code"
DRIVE_DIR = "/content/drive/MyDrive/toyllm2"

BASE_RUN = "sft"         # 起点：runs/<BASE_RUN>/model
RUN_TRAIN = True         # DPO，默认把训练集过一遍
RUN_NAME = "dpo"
TRAIN_ARGS = []          # 额外的参数，比如 ["--beta", "0.5", "--lr", "1e-6"]
RUN_EXPERIMENTS = True   # 对照实验：beta取0.05、0.1、0.5，各训400步，共约40分钟

In [ ]:
import os, pathlib, shutil, subprocess, sys

def sh(cmd, cwd=None):
    """跑一条shell命令，边跑边打印，失败就停。"""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

import torch
assert torch.cuda.is_available(), "没有CUDA：菜单“代码执行程序 → 更改运行时类型”选GPU"
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability()
print(f"GPU    {p.name}，sm{cap[0]}{cap[1]}，显存{p.total_memory / 2**30:.1f}GiB")
print(f"torch  {torch.__version__}，CUDA {torch.version.cuda}，Python {sys.version.split()[0]}")
print(f"CPU    {os.cpu_count()}个线程")
import numpy, pyarrow, safetensors, tokenizers   # 都是Colab自带的，缺了这里会报错

## 挂载Drive，拉代码

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_DIR)
RUNS = DRIVE / "runs"
BASE = RUNS / BASE_RUN / "model"
assert (BASE / "model.safetensors").exists(), f"Drive上没有{BASE}，先跑第七章"

# 稀疏克隆：整个仓库有好几本书，这里只取第八章的代码。
WORK = pathlib.Path("/content/vibe-writing")
if not WORK.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git sparse-checkout set --no-cone /{SUBDIR}/book/ch08/", cwd=WORK)
    sh("git checkout", cwd=WORK)
else:
    sh("git pull --ff-only", cwd=WORK)
sh("git log -1 --format='%h %ad %s' --date=short", cwd=WORK)
CODE = WORK / SUBDIR

def run(*args):
    """在code/下跑book/ch08/里的一个脚本，输出边跑边打印。"""
    cmd = [sys.executable, "-u", f"book/ch08/{args[0]}", *map(str, args[1:])]
    print("$ python", " ".join(cmd[2:]), flush=True)
    subprocess.run(cmd, cwd=CODE, check=True)

## 准备数据

Drive上已经有`toyllm2/data/dpo/`就直接复制到本地盘；没有就现做一份（下载加分词，几分钟），再存回Drive。

In [ ]:
DATA = pathlib.Path("/content/data/dpo")
SAVED = DRIVE / "data" / "dpo"
FILES = [f"{s}{x}" for s in ("train", "val")
         for x in (".bin", "_mask.bin", "_offsets.npy")]
if all((SAVED / f).exists() for f in FILES):
    DATA.mkdir(parents=True, exist_ok=True)
    for f in FILES:
        if not (DATA / f).exists() or (DATA / f).stat().st_size != (SAVED / f).stat().st_size:
            sh(f"cp '{SAVED / f}' '{DATA / f}'")
else:
    run("prepare_dpo.py", "--raw", "/content/data/ultrafeedback", "--out", DATA,
        "--tokenizer", BASE / "tokenizer.json")
    SAVED.mkdir(parents=True, exist_ok=True)
    for f in FILES:
        sh(f"cp '{DATA / f}' '{SAVED / f}'")
for f in FILES:
    print(f"{f:<18}{(DATA / f).stat().st_size:>16,} 字节")

## DPO

断了重连以后，再跑一遍这个格子就会接着训。训完存成模型目录：`toyllm2/runs/<名字>/model/`。

In [ ]:
if RUN_TRAIN and (RUNS / RUN_NAME / "model").exists():
    print(f"{RUN_NAME}已经训完，跳过")
elif RUN_TRAIN:
    run("dpo.py", "--name", RUN_NAME, "--init", BASE, "--data", DATA, "--runs", RUNS,
        "--resume", "--export", *TRAIN_ARGS)

## 对照实验：beta

三个`beta`各训400步，其余和完整训练一样。学习率按400步重新排，所以`beta`为0.1的那一次也要另训，不能拿完整训练的前400步来比。

每个实验的记录写在Drive的`toyllm2/runs/exp-beta*/`下，带着`--resume`，断了重跑这个格子就接着来，训完的不会重训。

In [ ]:
BETAS = [0.05, 0.1, 0.5]
SHORT = ["--steps", 400, "--warmup", 40, "--eval-every", 100, "--sample-every", 400,
         "--ckpt-every", 100, "--export"]

if RUN_EXPERIMENTS:
    for beta in BETAS:
        name = f"exp-beta{beta}"
        if (RUNS / name / "model").exists():
            print(f"{name}已经训完，跳过")
            continue
        run("dpo.py", "--name", name, "--init", BASE, "--data", DATA, "--runs", RUNS,
            "--beta", beta, "--resume", *SHORT)

## 问几个问题：DPO之前和之后

In [ ]:
run("chat.py", "--model", BASE, RUNS / RUN_NAME / "model")